# U.S Rates & Breakeven Inflation Analytics Engine
### Joint Term Structure Modeling, DKW (2018) Liquidity Wedge & Basel III / FRTB Market Risk

This notebook demonstrates the institutional research workflow using the `rate_engine` package.
It estimates a joint 4-parameter Nelson-Siegel term structure under **Diebold-Li (2006) Stuctural Basis Invariance**,
adjusts for latent TIPS liquidity frictions following **D'Amico, Kim, and Wei (DKW 2018)**, and sizes
duration- and beta-neutral breakeven boxes under strict **SR 11-7 Model Risk Governance**.

In [2]:
# Environment Setup & Modular Package Imports
# If running in a fresh Google Colab environment, uncomment the lines below:
# !git clone https://github.com/sungyup-jung/us-rates-breakeven-engine.git
# %cd us-rates-breakeven-engine
# !pip install -e

import datetime
from IPython.display import Markdown, display
import numpy as np

from rates_engine.data import FREDMarketDataLoader
from rates_engine.curves import YieldCurve, AdaptiveCurveSelector
from rates_engine.frictions import DKWEconometricPriors, DynamicMarketFrictions
from rates_engine.decompositor import InflationDecompositor, CashBondDiscountEngine
from rates_engine.risk import BreakevenTradePricer, HistoricalMarketRiskEngine, CurveSpreadPricer
from rates_engine.visualizer import render_rates_inflation_dashboard, MarkdownReportGenerator


ModuleNotFoundError: No module named 'rates_engine'

In [ ]:
# Live FRED Ingestion & Diebold-Li (2006) Curve Calibration
(
    nom_mats, nom_yields, tips_mats, tips_yields,
    survey_mats, survey_cpi_exp, dyn_seasonal_factors,
    nom_10y_s, tips_10y_s, nom_30y_s, tips_30y_s,
    cpi_nsa, stress_idx, sofr_val, tgcr_val, settle_date
) = FREDMarketDataLoader.fetch_latest_market_data()

eval_grid = np.array([2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])

# Calibrate Curves via Closed-Form WLS
# Downweight the 20Y nominal supply concession: w_20Y = 0.05
nom_weights = np.ones_like(nom_yields)
nom_weights[np.isclose(nom_mats, 20.0)] = 0.05
nom_curve = YieldCurve(nom_mats, nom_yields, weights=nom_weights)

# TIPS Curve: Invariant decay parameter tau1 ensures closure under subtraction
tips_curve = YieldCurve(tips_mats, tips_yields)

# Live Benchmark 10Y Par Repricing Check
actual_10y_yield = float(nom_yields[np.isclose(nom_mats, 10.0)][0])
semi_coupon = (actual_10y_yield / 2.0) * 100.0
live_cf_sched = np.array([semi_coupon] * 19 + [100.0 + semi_coupon])
live_cf_tenors = np.arange(0.5, 10.5, 0.5)
live_pv = CashBondDiscountEngine.bond_price_from_zero_curve(live_cf_sched, live_cf_tenors, nom_curve)
repricing_err_cents = (live_pv - 100.0) * 100.0

print("=== LIVE MODEL VERIFICATION ===")
print(f"Settlement Date                : {settle_date}")
print(f"FRED 10Y Benchmark Par Yield   : {actual_10y_yield * 100.0:.3f}%")
print(f"Model Park Repricing PV        : USD {live_pv:.4f} per $100 Par")
print(f"Fitting Concession Error       : {repricing_err_cents:+.2f} cents (Gate: +/- 5.0c)")

NameError: name 'FREDMarketDataLoader' is not defined

In [ ]:
# Frictions, Inflation Decomposition & Portfolio Risk Execution
# 1. Dynamic Liquidity Wedge (DKW 2018 Priors)
dkw_priors = DKWEconometricPriors()
dynamic_liq_wedge, sigma_liq_wedge = DynamicMarketFrictions.compute_dynamic_liquidity_wedge(
    maturities=eval_grid,
    financial_stress_idx=stress_idx,
    priors=dkw_priors
)

# 2. Inflation Decomposition & Latent IRP Extraction
decompositor = InflationDecompositor(nom_curve, tips_curve)
df_decomp, key_metrics = decompositor.decompose(
    eval_grid = eval_grid,
    survey_mats = survey_mats,
    survey_cpi_exp = survey_cpi_exp,
    liquidity_wedge_bps = dynamic_liq_wedge,
    sigma_wedge_bps = sigma_liq_wedge,
    seasonal_factors = dyn_seasonal_factors,
    current_month = datetime.datetime.now().month
)

# 3. Empirical Hedge Betas & Indexation (SR 11-7 Non-Degenerate OLS)
exec_params = DynamicMarketFrictions.compute_all_parameters(
    nom_10y_series = nom_10y_s,
    tips_10y_series = tips_10y_s,
    nom_30y_series = nom_30y_s,
    tips_30y_series = tips_30y_s,
    cpi_nsa_series = cpi_nsa,
    sofr_rate = sofr_val,
    tgcr_rate = tgcr_val,
    settlement_date = settle_date
)

# 4. Trade Structuring & FRTB Regulatory Market Risk Engine
pricer_10y = BreakevenTradePricer(
    nom_par_notional = 100_000_000,
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    tenor = 10.0,
    cif = exec_params['CIF'],
    beta_tips = exec_params['Beta_TIPS_10Y'],
    repo_nom_bps = exec_params['Repo_Nominal_Bps'],
    repo_tips_bps = exec_params['Repo_TIPS_Bps']
)
trade_structure_10y = pricer_10y.calculate_trade_structure()

bic_selection = AdaptiveCurveSelector.evaluate_model_selection(nom_mats, nom_yields)
market_risk = HistoricalMarketRiskEngine.evaluate_portfolio_var(
    nom_10y_series = nom_10y_s,
    tips_10y_series = tips_10y_s,
    pricer = pricer_10y,
    holding_period_days = 10,
    confidence_level = 0.99
)
curve_box_30y = CurveSpreadPricer.size_10s30s_box(
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    cif = exec_params['CIF'],
    beta_10y = exec_params['Beta_TIPS_10Y'],
    beta_30y = exec_params['Beta_TIPS_30Y'],
    target_10y_notional = 100_000_000.0
)

In [ ]:
# Render Interactive Dashboard & Research Note
render_rates_inflation_dashboard(
    df = df_decomp,
    metrics = key_metrics,
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    pricer_10y = pricer_10y,
    settlement_date = settle_date
)

report_md = MarkdownReportGenerator.generate_markdown(
    df_decomp =df_decomp,
    metrics = key_metrics,
    bic_selection = bic_selection,
    market_risk = market_risk,
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    exec_params = exec_params,
    trade_structure = trade_structure_10y,
    pricer_10y = pricer_10y,
    curve_box_30y = curve_box_30y,
    settle_date = settle_date,
    export_filename = "TERM_STRUCTURE_ANALYTICS_OUTPUT.md"
)
display(Markdown(report_md))